# Fine-Tune Qwen3-8B into a PII Redaction Engine on Crusoe

This notebook turns [Qwen3 8B](https://huggingface.co/Qwen/Qwen3-8B) into a specialist that finds and masks personally identifiable information (PII) in financial documents, end to end, using two Crusoe Intelligence Foundry products:

- [Serverless Fine-Tuning](https://docs.crusoecloud.com/serverless-fine-tuning/overview) ([launch blog](https://www.crusoe.ai/resources/blog/crusoe-introduces-serverless-fine-tuning)): upload a dataset, submit a job, get LoRA adapter checkpoints back. No GPUs to manage.
- [Self-Serve Deployments](https://docs.crusoecloud.com/self-serve-deployments/overview) ([launch blog](https://www.crusoe.ai/resources/blog/crusoe-self-serve-deployments)): one click turns your checkpoint into a dedicated, single-tenant inference endpoint.

**Why PII redaction?** It is the canonical workload you cannot send to a third-party model API, because the input is the sensitive data itself. Regulated teams in finance, healthcare, and insurance need the model to come to the data. Fine-tuning an open model and serving it inside infrastructure you control is the whole point of this stack.

By the end you will have:

1. built a training set from a public Hugging Face dataset,
2. fine-tuned Qwen3 8B with LoRA on Crusoe Serverless Fine-Tuning,
3. deployed the checkpoint as a dedicated endpoint, and
4. measured it against a general-purpose 70B model on entity level F1.

## `Step 1`: Introduction

Crusoe's Managed AI products live under one roof called **Crusoe Intelligence Foundry**. Three pieces matter here:

![PII redaction pipeline on Crusoe: raw documents, Serverless Fine-Tuning, Self-Serve Deployment, redacted output](assets/pii-redaction-pipeline.svg)

- **Serverless Fine-Tuning**: you upload a dataset and submit a job; Crusoe provisions the GPUs, trains, checkpoints, and stores the results. You never see a VM.
- **Self-Serve Deployments**: one click turns a base model or your fine-tuned checkpoint into a dedicated, single-tenant inference endpoint with no rate limits.
- **Serverless Inference**: pay-per-token shared endpoints for popular open models. We will use one as the evaluation baseline in section 10.

The whole platform is **OpenAI-API-compatible**: the `openai` Python package is the only client you need, pointed at two base URLs.

| Base URL | What lives there |
|---|---|
| `https://api.intelligence.crusoecloud.com/v1` | `Control plane`: files, fine-tuning jobs, model registry |
| `https://api.inference.crusoecloud.com/v1` | `Data plane`: chat completions for serverless models and your deployments |

The full REST surface is documented in the [Fine-tuning API reference](https://docs.crusoecloud.com/api/managed-ai/#tag/Fine-tuning).

## `Step 2`: The base model: `Qwen3-8B`



[Qwen3 8B](https://huggingface.co/Qwen/Qwen3-8B) is an 8.2B-parameter dense model from the Qwen3 series, Apache 2.0 licensed, and a sweet spot for task specialization:

- **Fast, cheap training**: a 2-epoch LoRA pass over this notebook's dataset finishes in minutes, not hours.
- **Single-GPU serving**: an 8B dense model deploys on one GPU, the cheapest possible dedicated endpoint.
- **The right size for the job**: for format specialization like PII redaction, a well-trained small model matches much larger general models, at a fraction of the serving cost.

The recipe below is also model agnostic: swap `BASE_MODEL_NAME` in section 4.4 for any model in the fine-tuning registry and everything else stays the same.

## `Step 3`: How Serverless Fine-Tuning works: `LoRA` on a quantized base



All training on Serverless Fine-Tuning is **LoRA (Low-Rank Adaptation)**. Instead of updating all of the base model's weights (expensive, slow, easy to catastrophically overfit), LoRA freezes the base model and trains two small low-rank matrices, **A** and **B**, alongside each targeted weight matrix. The effective weight becomes `W + B*A`, where `B*A` has rank `r` (you set `lora_rank` yourself later).

![LoRA](./assets/lora.svg)

**Where quantization fits.** The frozen base weights only need to be read, never updated, so they can sit in memory in a compressed format (FP8 checkpoints, for example, are half the bytes of BF16), while the trainable adapter matrices stay in higher precision because gradients flow through them. Training LoRA adapters on top of a quantized frozen base is the idea behind QLoRA: quantization shrinks the frozen part, LoRA shrinks the trainable part.

What this means for you in practice:

- **The artifact you get back is tiny**: tens to hundreds of MB of adapter weights, not a multi-hundred-GB model copy. You can download it, version it, and own it.
- **Training is fast and cheap**: only the adapter parameters get gradients, and you are billed per training token, not per GPU-hour.
- **Quality is excellent for specialization**: teaching format, domain vocabulary, and task behavior, exactly what PII redaction needs. It will not teach the model new world knowledge; for that, look at RAG.

**When should you fine-tune at all?**

| Your problem | Reach for |
|---|---|
| Model does not know facts about your private data | RAG / retrieval |
| Model knows how, but output format, style, or reliability is wrong | Prompting, then **fine-tuning** when prompting plateaus |
| Data is too sensitive to send to a third-party model API | **Fine-tune an open model in your own environment** (this notebook) |

## `Step 4`: Setup



### 4.1 Get your Crusoe API key

1. Log in to the [Crusoe Console](https://console.crusoecloud.com/).
2. Go to **Organization settings -> Security -> [Inference API keys](https://console.crusoecloud.com/security/inference-api-keys)**.
3. Create a key and copy it.

### 4.2 Configure your environment

```bash
cp .env.example .env    # then paste your key into CRUSOE_API_KEY

# with uv (recommended, https://docs.astral.sh/uv/):
uv venv && uv pip install -r requirements.txt

# or with plain pip:
python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt
```

Helper functions for validation, plotting, and evaluation live in [`utils.py`](utils.py) so the cells below stay focused on the workflow.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import httpx
from dotenv import load_dotenv
from openai import OpenAI

import utils

load_dotenv()
CRUSOE_API_KEY = os.environ["CRUSOE_API_KEY"]

# Control plane: files, fine-tuning jobs, model registry
INTELLIGENCE_BASE_URL = "https://api.intelligence.crusoecloud.com/v1"
# Data plane: chat completions for serverless models and your deployments
INFERENCE_BASE_URL = "https://api.inference.crusoecloud.com/v1"

client = OpenAI(
    api_key=CRUSOE_API_KEY,
    base_url=INTELLIGENCE_BASE_URL,
    http_client=httpx.Client(proxy=None, trust_env=False),
)

### 4.3 Sanity check: list the models you can fine-tune

The model registry tells you which base models support fine-tuning right now.

In [ ]:
models = client.models.list().data
fine_tunable = [m for m in models if getattr(m, "fine_tuning_available", False)]

print(f"{len(fine_tunable)} fine-tunable base models available:\n")
print(f"{'Model ID':<52}  HuggingFace name")
print("-" * 92)
for m in sorted(fine_tunable, key=lambda m: m.model_name.lower()):
    print(f"{m.id:<52}  {m.model_name}")

### 4.4 Choose the base model

Fine-tuning jobs take the registry model ID, so resolve it from the HuggingFace-style name.

In [ ]:
BASE_MODEL_NAME = "Qwen/Qwen3-8B"
BASE_MODEL_ID = next(m.id for m in fine_tunable if m.model_name == BASE_MODEL_NAME)
print(f"Base model    :    {BASE_MODEL_NAME}")
print(f"Base model ID :    {BASE_MODEL_ID}")

## `Step 5`: Uploading the `dataset`



We train on [gretelai/synthetic_pii_finance_multilingual](https://huggingface.co/datasets/gretelai/synthetic_pii_finance_multilingual): about 56,000 **fully synthetic** financial documents, each with labeled PII spans covering 29 entity types (names, companies, account numbers, addresses, emails, and more). Synthetic data means no real person's information appears anywhere in this walkthrough.

[`data/prepare_dataset.py`](data/prepare_dataset.py) downloads the dataset from the Hugging Face Hub and converts it into the chat format the fine-tuning service expects, one JSON object per line:

```json
{"messages": [
  {"role": "system", "content": "You are a PII redaction engine..."},
  {"role": "user", "content": "<the raw document>"},
  {"role": "assistant", "content": "{\"redacted_text\": ..., \"entities\": [...]}"}
]}
```

The script keeps short English documents with at least two PII spans and a quality score of 80 or higher, then writes three files:

| File | Rows | Used for |
|---|---|---|
| `data/train.jsonl` | 2,500 | gradient updates |
| `data/validation.jsonl` | 150 | validation loss during training |
| `data/test.jsonl` | 100 | held out locally for the evaluation in section 10 |

The assistant turn is a JSON object with the redacted document and the entity list. Teaching the model to emit that contract reliably is exactly the kind of format specialization LoRA excels at.

In [ ]:
!python ./data/prepare_dataset.py

### Validate the data


In [ ]:
DATA_DIR = Path("data")

train_rows = utils.load_jsonl(DATA_DIR / "train.jsonl")
val_rows = utils.load_jsonl(DATA_DIR / "validation.jsonl")
test_rows = utils.load_jsonl(DATA_DIR / "test.jsonl")
print(f"train {len(train_rows)}, validation {len(val_rows)}, held-out test {len(test_rows)}")

In [ ]:
# Checking one of the training rows to see what the data looks like (e.g. row number 11)
system_msg, user_msg, assistant_msg = train_rows[11]["messages"]
target = json.loads(assistant_msg["content"])

print("================================")
print("USER (raw document)")
print("================================")
print(user_msg["content"])
print("================================")
print("ASSISTANT (training target)")
print("================================")
print(target["redacted_text"])
print("================================")
print("Entities:")
print("================================")
for e in target["entities"]:
    print(f"  {e['type']:<20} {e['value']}")

Now that we have over data set we can now upload it. 

Files should go to the control plane with the standard OpenAI files API, `purpose="fine-tune"`.

In [ ]:
with open(DATA_DIR / "train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")

with open(DATA_DIR / "validation.jsonl", "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")

print(f"Training file:   {train_file.id}  ({train_file.bytes:,} bytes)")
print(f"Validation file: {val_file.id}  ({val_file.bytes:,} bytes)")

## `Step 6`: Launch the fine-tuning job



Every hyperparameter is optional (the service picks sensible `auto` values), but the knobs worth knowing, straight from the [API schema](https://docs.crusoecloud.com/api/managed-ai/#tag/Fine-tuning):

| Hyperparameter | Default | What it does |
|---|---|---|
| `n_epochs` | `auto` | Full passes over the dataset (1-50). More epochs, more specialization, more overfitting risk. |
| `batch_size` | `auto` | Examples per update; power of 2, 16-1024. |
| `learning_rate` | `auto` | Absolute learning rate. |
| `lr_scheduler` | `cosine` | `cosine`, `linear`, `constant`, `constant_with_warmup`. |
| `lora_rank` | `auto` | The r in LoRA, the adapter capacity. Powers of 2, 2-256. |
| `lora_alpha` | `auto` | LoRA scaling; a common heuristic is 2x the rank. |
| `lora_dropout` | off | Dropout on adapter activations, regularization. |
| `early_stopping_patience` | off | Stop after N evaluations without improvement. |
| `checkpoint_steps` | `100` | Save a checkpoint every N update steps. |
| `eval_steps_per_epoch` | `4` | Validation passes per epoch. |
| `top_k_gating` | model default | MoE models only: experts activated per token. |

PII redaction is format specialization on a well-represented domain, so a modest adapter and two passes over 2,500 examples is plenty.

In [ ]:
HYPERPARAMETERS = {
    "n_epochs": 2,          
    "batch_size": 16,       
    "learning_rate": 1e-4,  
    "lora_rank": 16,        
    "lora_alpha": 32,       
}

job = client.fine_tuning.jobs.create(
    model=BASE_MODEL_ID,
    training_file=train_file.id,
    validation_file=val_file.id,
    suffix="pii-redaction-qwen",  
    method={
        "type": "supervised",
        "supervised": {"hyperparameters": HYPERPARAMETERS},
    },
)

print(f"Job ID: {job.id}")
print(f"Status: {job.status}")
print("\nWatch it live in the console:")
print(f"  https://console.crusoecloud.com/foundry/fine-tuning/jobs/{job.id}/overview")

### What happens now, behind the scenes

You handed Crusoe a job spec. The platform validates your files, queues the job, provisions GPUs, streams your data through the trainer, and saves LoRA checkpoints with metrics to object storage at every `checkpoint_steps` interval.

![SFT](./assets/SFT-state.png)

Billing is **per training token processed**, so you pay for results, not idle GPUs. Details in the [Serverless Fine-Tuning docs](https://docs.crusoecloud.com/serverless-fine-tuning/overview).

## `Step 7`: Monitor training



Poll the job until it reaches a terminal state. The cell is interrupt-safe: stop it and re-run any time, it just resumes polling.

In [ ]:
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}

def wait_for_job(job_id, poll_seconds=15):
    last_status, started = None, time.time()
    while True:
        j = client.fine_tuning.jobs.retrieve(job_id)
        if j.status != last_status:
            print(f"[{int(time.time() - started):>4}s] status -> {j.status}")
            last_status = j.status
        if j.status in TERMINAL_STATUSES:
            return j
        time.sleep(poll_seconds)

job = wait_for_job(job.id)
assert job.status == "succeeded", f"Job ended as {job.status}, see events below or the console"
print("\nTraining complete")

### Job events: the audit trail

Every state transition and trainer message is recorded, useful for debugging failed jobs and for compliance trails.

In [ ]:
events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id).data
for e in reversed(events):
    ts = time.strftime("%H:%M:%S", time.localtime(getattr(e, "created_at", 0)))
    print(f"{ts}  [{getattr(e, 'level', 'info'):<5}]  {e.message}")

### Training curves

The `/fine_tuning/jobs/{id}/metrics` endpoint returns time series for training loss (`ev_loss`), validation loss (`ev_eval_loss`), and tokens processed. 

You can run this cell while the job is still training to watch the curves fill in.

In [ ]:
resp = httpx.get(
    f"{INTELLIGENCE_BASE_URL}/fine_tuning/jobs/{job.id}/metrics",
    headers={"Authorization": f"Bearer {CRUSOE_API_KEY}"},
    timeout=30,
)
resp.raise_for_status()

utils.plot_training_curves(resp.json(), title="Qwen3 8B LoRA fine-tune, loss curves")

### Checkpoints: your trained artifacts

Each checkpoint is a complete, deployable LoRA adapter with its metrics, so you can pick an earlier step if the final one overfit.

In [ ]:
checkpoints = client.fine_tuning.jobs.checkpoints.list(job.id).data
print(f"{len(checkpoints)} checkpoint(s):\n")
for c in sorted(checkpoints, key=lambda c: c.step_number or 0):
    metrics = getattr(c, "metrics", None)
    tloss = getattr(metrics, "train_loss", None) if metrics else None
    vloss = getattr(metrics, "valid_loss", None) if metrics else None
    print(f"step {c.step_number:>5}  train_loss={tloss}  valid_loss={vloss}")
    print(f"           model: {getattr(c, 'fine_tuned_model_id', None)}")

In [ ]:
# The final checkpoint registers as a model shortly after the job succeeds
deadline = time.time() + 180
while not client.fine_tuning.jobs.retrieve(job.id).fine_tuned_model and time.time() < deadline:
    time.sleep(5)

job = client.fine_tuning.jobs.retrieve(job.id)
FINE_TUNED_MODEL = job.fine_tuned_model
assert FINE_TUNED_MODEL, "Model not registered yet, re-run this cell in a minute"
print(f"Fine-tuned model ID: {FINE_TUNED_MODEL}")

### `Optional`: download your adapter

The adapter weights are yours. A few hundred MB of safetensors you can version, audit, or serve anywhere else.

In [ ]:
DOWNLOAD_ADAPTER = True  # True/False to fetch/not fetch the LoRA weights

if DOWNLOAD_ADAPTER:
    out_dir = Path("outputs")
    out_dir.mkdir(exist_ok=True)
    zip_path = out_dir / f"{FINE_TUNED_MODEL}.zip"
    with httpx.stream(
        "GET",
        f"{INTELLIGENCE_BASE_URL}/models/{FINE_TUNED_MODEL}/download",
        headers={"Authorization": f"Bearer {CRUSOE_API_KEY}"},
        timeout=300,
    ) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_bytes(8192):
                f.write(chunk)
    print(f"Saved {zip_path} ({zip_path.stat().st_size:,} bytes)")
else:
    print("Skipped, set DOWNLOAD_ADAPTER = True to fetch the LoRA weights")

## `Step 8`: Deploy the fine-tuned model

**This step happens in the UI (as of today), not in code.** Creating a [Self-Serve Deployment](https://docs.crusoecloud.com/self-serve-deployments/overview) is done through the Crusoe Console UI:

1. Open the [Intelligence Foundry console](https://console.crusoecloud.com/foundry/deployments) and make sure you are in **Intelligence Foundry** (top-left switcher).
2. Open your **fine-tuning job's page**, then the **three-dot menu**, then **Deploy**.
3. Choose an **optimization profile** (see table below), set **replica count = 1**, and give the deployment an **alias** you will recognize, for example `pii-redaction-demo`. This alias is what you pass as the `model` parameter in API calls.

| | `Responsiveness` | `Throughput` |
|---|---|---|
| Optimizes for | Time-to-first-token, low latency | Tokens per second per dollar |
| Pick it when | User-facing products | Batch pipelines, like bulk document redaction |

4. Click create, then wait for the status to go **Creating -> Ready**
More in the [Self-Serve Deployments launch blog](https://www.crusoe.ai/resources/blog/crusoe-self-serve-deployments).

Once the deployment is over, paste the alias below:

In [ ]:
# DEPLOYMENT_ALIAS = <PASTE-YOUR-DEPLOYMENT-ALIAS-HERE>
DEPLOYMENT_ALIAS = "test-qwen3.5-9B-ft"
print(f"Deployment alias: {DEPLOYMENT_ALIAS}")

## `Step 9`: Inference


In [ ]:
inference = OpenAI(
    api_key=CRUSOE_API_KEY,
    base_url=INFERENCE_BASE_URL,
    http_client=httpx.Client(proxy=None, trust_env=False),
)

demo_document = """Subject: Wire transfer confirmation

Hi Suman,

The transfer of $18,400 to Meridian Holdings LLC was initiated on March 3, 2026
from account 4432-889912. Please confirm receipt with our treasury desk at
treasury@meridianholdings.com, or call Daniel Okafor at (415) 555-0173.

Thanks,
Elena Vasquez
Senior Treasury Analyst"""

response = inference.chat.completions.create(
    model=DEPLOYMENT_ALIAS,
    messages=[
        {"role": "system", "content": utils.SYSTEM_PROMPT},
        {"role": "user", "content": demo_document},
    ],
    temperature=0,
)

prediction = utils.parse_prediction(response.choices[0].message.content)
print(prediction["redacted_text"])
print("\nEntities found:")
for e in prediction["entities"]:
    print(f"  {e['type']:<18} {e['value']}")

### Streaming

The endpoint supports streaming, the natural fit when redaction runs behind an interactive review UI.

In [ ]:
stream = inference.chat.completions.create(
    model=DEPLOYMENT_ALIAS,
    messages=[
        {"role": "system", "content": utils.SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Reminder: send the signed NDA to Marcus Webb at m.webb@stratos.io "
            "before our call on Friday, June 12, 2026. His cell is 917-555-0244."
        )},
    ],
    temperature=0,
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## `Step 10`: Evaluate the fine-tuned model vs a general-purpose model



Does fine-tuning actually buy anything over prompting a big general model? We run both over the 100 held-out test documents with the identical system prompt and score them at the entity level:

- **Detection**: the model found the PII string, whatever type it assigned. Precision, recall, and F1 over (document, value) pairs.
- **Typed**: the value and its type label both match the reference. This is where format and taxonomy training shows.
- **Bad JSON**: replies that could not be parsed into the output contract at all. In a production redaction pipeline this is the metric that pages someone.

The baseline is Llama 3.3 70B Instruct on Crusoe Serverless Inference (fine-tunable registry models are not offered on shared serverless endpoints, so the fine-tune-and-deploy path is how you run them). Scoring lives in `utils.score_entities`; the loop below is just API calls.

In [ ]:
BASELINE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"  # serverless, pay per token
EVAL_N = 25  # bump toward 100 for a more confident comparison

contenders = {
    "Fine-tuned Qwen3 8B": DEPLOYMENT_ALIAS,
    "Llama 3.3 70B (serverless)": BASELINE_MODEL,
}

def predict(model_name, document):
    r = inference.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": utils.SYSTEM_PROMPT},
            {"role": "user", "content": document},
        ],
        temperature=0,
        max_tokens=1500,
    )
    return utils.parse_prediction(r.choices[0].message.content)

scores = {label: [] for label in contenders}
parse_failures = {label: 0 for label in contenders}

for row in test_rows[:EVAL_N]:
    document = row["messages"][1]["content"]
    reference = json.loads(row["messages"][2]["content"])["entities"]
    for label, model_name in contenders.items():
        pred = predict(model_name, document)
        if pred is None:
            parse_failures[label] += 1
            pred = {"entities": []}
        scores[label].append(utils.score_entities(reference, pred["entities"]))
    print(".", end="", flush=True)
print(f" evaluated {min(EVAL_N, len(test_rows))} documents")

In [ ]:
summaries = {label: utils.aggregate_scores(rows) for label, rows in scores.items()}
utils.report_eval(summaries, parse_failures, min(EVAL_N, len(test_rows)))

In [ ]:
# Qualitative peek at one document
row = test_rows[0]
document = row["messages"][1]["content"]
reference = json.loads(row["messages"][2]["content"])["entities"]

print(document[:400])
print(f"\nReference: {len(reference)} entities")
for label, model_name in contenders.items():
    pred = predict(model_name, document) or {"entities": []}
    print(f"\n{label}: {len(pred['entities'])} entities")
    for e in pred["entities"][:6]:
        print(f"  {e['type']:<18} {e['value']}")

Even where a big general model matches the quality, the specialized model wins on unit economics and posture for a high-volume narrow task:

| | Fine-tuned Qwen3 8B (dedicated) | General 70B (serverless) |
|---|---|---|
| Pricing model | Flat per GPU-hour, no rate limits | Per token, forever |
| At sustained high volume | Flat cost amortizes; past break-even, tokens are effectively free | Cost scales linearly with tokens |
| Data path | Single-tenant endpoint in your environment | Shared endpoint |
| Latency | Dedicated capacity, no noisy neighbors | Shared capacity |

For redaction pipelines that chew through millions of documents, sustained volume is the definition of the workload, and the single-tenant data path is usually a compliance requirement rather than a preference.

## `Step 11`: Clean up

Two things can keep billing after you close this notebook:

1. **The deployment** bills per GPU-hour until deleted: [Deployments page](https://console.crusoecloud.com/foundry/deployments), three-dot menu, **Delete**.
2. **Uploaded dataset files** are cheap but tidy people delete them.

Fine-tuned model checkpoints stay in the registry so you can redeploy any time.

In [ ]:
CLEAN_UP_FILES = False  # flip to True to delete the uploaded dataset files

if CLEAN_UP_FILES:
    for f_obj in (train_file, val_file):
        client.files.delete(f_obj.id)
        print(f"Deleted {f_obj.id}")
else:
    print("Skipped, set CLEAN_UP_FILES = True to delete the uploaded files")

## What next!



- [Serverless Fine-Tuning documentation](https://docs.crusoecloud.com/serverless-fine-tuning/overview) and [launch blog](https://www.crusoe.ai/resources/blog/crusoe-introduces-serverless-fine-tuning)
- [Self-Serve Deployments documentation](https://docs.crusoecloud.com/self-serve-deployments/overview) and [launch blog](https://www.crusoe.ai/resources/blog/crusoe-self-serve-deployments)
- [Fine-tuning API reference](https://docs.crusoecloud.com/api/managed-ai/#tag/Fine-tuning)
- Try the same recipe on your own documents: swap `data/prepare_dataset.py` for a script that reads your redaction logs, and everything downstream stays identical.